# Copy Gold views → Staging_Gold (keep both)

Creates the same performance views in `Staging_Gold` while keeping them in `Gold`.
Source of truth remains `Gold.rpt_unified_ad_performance` / Meta / Google rpt tables.



In [ ]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS Gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS Staging_Gold")
print("schemas ready")
print("Gold tables/views sample:", spark.sql("SHOW VIEWS IN Gold").collect())



In [ ]:
# Recreate Gold views (ensure present)
spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_ad_performance AS
SELECT * FROM Gold.rpt_unified_ad_performance
''')

spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_unified_ad_performance AS
SELECT * FROM Gold.rpt_unified_ad_performance
''')

spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_meta_ad_performance AS
SELECT * FROM Gold.rpt_meta_ad_performance_daily
''')

spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_google_ad_performance AS
SELECT * FROM Gold.rpt_google_ad_performance_daily
''')

spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_adset_performance AS
SELECT
  platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  adset_id, MAX(adset_name) AS adset_name, MAX(adset_status) AS adset_status,
  MAX(optimization_goal) AS optimization_goal,
  MAX(age_range) AS age_range, MAX(geo_cities) AS geo_cities, MAX(geo_regions) AS geo_regions,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT ad_id) AS ad_count
FROM Gold.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id, adset_id
''')

spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_campaign_performance AS
SELECT
  platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT adset_id) AS adset_count, COUNT(DISTINCT ad_id) AS ad_count
FROM Gold.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id
''')

# optional by-level convenience view if useful
spark.sql('''
CREATE OR REPLACE VIEW Gold.vw_performance_by_level AS
SELECT 'ad' AS level, platform, full_date, account_id, account_name,
  campaign_id, campaign_name, adset_id, adset_name, ad_id, ad_name,
  spend, clicks, impressions, leads, cpc, cpm, gold_processed_at
FROM Gold.rpt_unified_ad_performance
UNION ALL
SELECT 'adset' AS level, platform, full_date, account_id, account_name,
  campaign_id, campaign_name, adset_id, adset_name,
  CAST(NULL AS STRING) AS ad_id, CAST(NULL AS STRING) AS ad_name,
  spend, clicks, impressions, leads, cpc, cpm, CAST(NULL AS TIMESTAMP) AS gold_processed_at
FROM Gold.vw_adset_performance
UNION ALL
SELECT 'campaign' AS level, platform, full_date, account_id, account_name,
  campaign_id, campaign_name,
  CAST(NULL AS STRING) AS adset_id, CAST(NULL AS STRING) AS adset_name,
  CAST(NULL AS STRING) AS ad_id, CAST(NULL AS STRING) AS ad_name,
  spend, clicks, impressions, leads, cpc, cpm, CAST(NULL AS TIMESTAMP) AS gold_processed_at
FROM Gold.vw_campaign_performance
''')
print("[OK] Gold views refreshed")



In [ ]:
# Mirror same views into Staging_Gold (pointing at Gold source tables)
spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_ad_performance AS
SELECT * FROM Gold.rpt_unified_ad_performance
''')

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_unified_ad_performance AS
SELECT * FROM Gold.rpt_unified_ad_performance
''')

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_meta_ad_performance AS
SELECT * FROM Gold.rpt_meta_ad_performance_daily
''')

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_google_ad_performance AS
SELECT * FROM Gold.rpt_google_ad_performance_daily
''')

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_adset_performance AS
SELECT
  platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  adset_id, MAX(adset_name) AS adset_name, MAX(adset_status) AS adset_status,
  MAX(optimization_goal) AS optimization_goal,
  MAX(age_range) AS age_range, MAX(geo_cities) AS geo_cities, MAX(geo_regions) AS geo_regions,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT ad_id) AS ad_count
FROM Gold.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id, adset_id
''')

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_campaign_performance AS
SELECT
  platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT adset_id) AS adset_count, COUNT(DISTINCT ad_id) AS ad_count
FROM Gold.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id
''')

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_performance_by_level AS
SELECT 'ad' AS level, platform, full_date, account_id, account_name,
  campaign_id, campaign_name, adset_id, adset_name, ad_id, ad_name,
  spend, clicks, impressions, leads, cpc, cpm, gold_processed_at
FROM Gold.rpt_unified_ad_performance
UNION ALL
SELECT 'adset' AS level, platform, full_date, account_id, account_name,
  campaign_id, campaign_name, adset_id, adset_name,
  CAST(NULL AS STRING) AS ad_id, CAST(NULL AS STRING) AS ad_name,
  spend, clicks, impressions, leads, cpc, cpm, CAST(NULL AS TIMESTAMP) AS gold_processed_at
FROM Staging_Gold.vw_adset_performance
UNION ALL
SELECT 'campaign' AS level, platform, full_date, account_id, account_name,
  campaign_id, campaign_name,
  CAST(NULL AS STRING) AS adset_id, CAST(NULL AS STRING) AS adset_name,
  CAST(NULL AS STRING) AS ad_id, CAST(NULL AS STRING) AS ad_name,
  spend, clicks, impressions, leads, cpc, cpm, CAST(NULL AS TIMESTAMP) AS gold_processed_at
FROM Staging_Gold.vw_campaign_performance
''')
print("[OK] Staging_Gold views created")



In [ ]:
# Validate both schemas
for schema in ["Gold", "Staging_Gold"]:
    print(f"\n=== {schema} VIEWS ===")
    spark.sql(f"SHOW VIEWS IN {schema}").show(50, truncate=False)

checks = [
    ("Gold.vw_ad_performance", "Staging_Gold.vw_ad_performance"),
    ("Gold.vw_adset_performance", "Staging_Gold.vw_adset_performance"),
    ("Gold.vw_campaign_performance", "Staging_Gold.vw_campaign_performance"),
    ("Gold.vw_unified_ad_performance", "Staging_Gold.vw_unified_ad_performance"),
]
for g, s in checks:
    gc = spark.table(g).count()
    sc = spark.table(s).count()
    print(f"{g}: {gc} | {s}: {sc} | match={gc==sc}")

print("STAGING_GOLD_VIEWS_COPY_COMPLETE")

